In [ ]:
## IMPORTS

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import ( train_test_split,
                                      GridSearchCV)

from sklearn.preprocessing import ( StandardScaler,
                                    OneHotEncoder, 
                                    PolynomialFeatures) 

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer

from sklearn.metrics import (accuracy_score,
                            precision_score,
                            recall_score,
                            f1_score,
                            confusion_matrix,
                            classification_report,
                            roc_auc_score,
                            ConfusionMatrixDisplay,
                            RocCurveDisplay,
                            PrecisionRecallDisplay)


In [ ]:
##importing data set
df = pd.read_csv("../data/machine_failure_dataset_cleaned.csv")

# **ANALYSING, CLEANING & EDA**

ANALYSIS

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
## There is a huge imbalance, **MUST DO CLASS=BALANCED**

df["machine_failure"].value_counts()

In [ ]:
df.isnull().sum()

CLEANING


In [ ]:
## SQL handled most of the cleaning, PYTHON will deal with NULL values(simple imputer, during pipeline building) and Outliers(during EDA)

In [ ]:
##REMOVING unwanted columns, product_ID is only an identifier and it has no predictive value.

df = df.drop("product_ID", axis=1)

**EXPLORITORY DATA ANALYSIS**

In [ ]:
## Separating numerical and categorical columns

numerical_columns = ["air_temperature_k",
                     "process_temperature_k",
                     "rotational_speed_rpm",
                     "torque_nm",
                     "tool_wear_min"
                     

]
categorical_columns = [
                        "product_type",
                        "tool_wear_failure",
                        "power_failure",
                        "overstrain_failure"
    
]

DISTRIBUTION
- help up understand how the data is distributed

In [ ]:
## Histogram (for numerical values)

for column in numerical_columns:
    plt.figure(figsize=(6,5))

    sns.histplot(
        data=df,
        x=column,
        hue="machine_failure",
        kde = True   
    )

    plt.title(f"{column} Distribution by class")
    plt.show()


Observation:

- air_temperature_k: shows a multimodal shape and failure are rare however there is a slight increan of failures forming a bell shape from 296.7 to 299
  
- process_temperature_k: multimodal and left-skewed, failures are rare however there is a slight increase of failures forming a bell shape from 307.5 to 309.5 of failures between
 
- rotational_speed_rpm: right skewed with a few outliers, failures are rare however they show a small left skewed shape in the beggining of the graph
- torque_nm: bell shape, left skewed a little bit, failures are still rare however they show a small bell shape from 40 to 90
- tool_wear_min: rectangle shape, right skewed a little, failures are rare howeever they form a small left skewed graph

In [ ]:
## Count plot(for categorical values)

for column in categorical_columns:
    plt.figure(figsize=(6,4))

    sns.countplot(
        data=df,
        x=column,
        hue="machine_failure"
    )

    plt.title(f"{column} Distribution by class")
    plt.show()




Observation:

- product_type: machine failure are less common for the product type H (High) and most common for product type L (Low), Medium (M) failures are less than product type L and more than product type H. 

- tool_wear_failure: The graph shows that most observations do not have a tool wear failure, only a small number of machine_failures occurred when there was no recorded tool wear failure, this suggests that tool wear failure is not common in the dataset.

- power_failure: The graph shows a few machine failure observations both when there is a power failure and when there is no power failure.

- overstrain_failure: The graph shows a few machine failure observations both when there is a overstrain failure and when there is no overstrain failure, indicating that machine failure occur when overstrain failure is present or not.


OUTLIER DETECTION
- While observing the distribution graphs we have discoverd that some numerical columns may contain outliers

In [ ]:
## Boxplot to see weather a feature contains outliers
for column in numerical_columns:
    plt.figure(figsize=(8,6))

    sns.boxplot(df[column])
    plt.title(f'Boxplot of {column}')
    plt.xlabel(f'{column}')
    plt.show()

Observation:

- air_temperature_k: all values are within the whiskers, indicating that no outliers were detected
  
- process_temperature_k: a few values are clustered close to each other appear below the lower whisker, since these values are not extremely isolated this could suggest a natural variation

- rotational_speed_rpm: a few values are clustered close to each other above the upper whisker, these values do not appear to be isolated and this suggest a natural variation

- torque_nm: there are a few values that appear below the lower whisker and above the upper whisker these values are clustered close to each other, however ther is a single value that is isolated way above the other values above the upper whisker. this value was identified as an artificially introduced value during data preparation, therefore it will me removed

- tool_wear_min: all values are within the whiskers, indicating that no outliers were detected

REMOVING OUTLIERS

In [ ]:
## Removing the artificially introduced torque value

df = df[df["torque_nm"] < 500]

In [ ]:
plt.figure(figsize=(14,3))
sns.boxplot(x=df["torque_nm"])
plt.title("Boxplot of torque_nm after removing the artificial outlier")
plt.show()

CORRELATION ANALYSIS

In [ ]:
## Correlation analysis for numerical values
## Heatmap 

plt.figure(figsize=(8,5))

sns.heatmap(
    df.corr(numeric_only = True),
    annot = True,
    cmap = "coolwarm"
)
plt.title("Correlation Heatmap")
plt.show()

Observation:

- process_temperature_k and air_temperature have a strong positive correlation of 0.8, this suggests that the two features are giving almost the same information so one could be considered for removal, process_temperature_k was chosen for removal because it has the lowest feature vs target correlation of - 0.016

- rotational_speed_rpm and torque_nm have a strong negative correlation of -0.88, rotational_speed_rpm could be considered for removal because it has i slightly weaker correlatrion with the target (0.13 vs 0.12)

- power_failure and overstrain_failure showed a positive correlation with the target, suggesting that they have a linear relationship and they will help the model make better predictions.

- the remaining features do not show any strong correlation and were therefore retained for model development

In [ ]:
## process_temperature_k removal

df = df.drop("process_temperature_k", axis=1)

In [ ]:
## rotational_speed_rpm removal 

df = df.drop("rotational_speed_rpm", axis=1)

In [ ]:
df.head()


# **PIPELINE BUILDING**

In [ ]:
## defining X and y

X = df.drop("machine_failure", axis=1)
y = df["machine_failure"]

TRAIN TEST SPLIT

In [ ]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42)

PIPELINE BUILDING and Preprocessing

In [ ]:
## RENEWING (Separating numerical and categorical columns)

numerical_columns = ["air_temperature_k",
                     "torque_nm",
                     "tool_wear_min"
]

categorical_columns = [
                        "product_type",
                        "tool_wear_failure",
                        "power_failure",
                        "overstrain_failure"
]

In [ ]:
numerical_pipeline = Pipeline([
   ("imputer", SimpleImputer(strategy= "median")),
   ("scaler", StandardScaler()),
   ("poly", PolynomialFeatures( include_bias=False))


])

In [ ]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy= "most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

PREPROCESSOR
Preprocessed data: prepared, cleaned, consistent and ready for machine learning to learn from

In [ ]:
preprocessor = ColumnTransformer([
    ("numerical", numerical_pipeline, numerical_columns),
    ("categorical", categorical_pipeline, categorical_columns)
])

LOGISTIC REGRESSION & GridSearchCV

In [ ]:
## parameters and model

models = {
    "LogisticRegression": {
        "model": LogisticRegression(class_weight="balanced", random_state=42),
        "params": {
            "model__C": [ 0.01, 0.05,0.1, 0.2],
            "model__penalty": ["l1", "l2"],
            "model__solver": ["liblinear"],
            "preprocessor__numerical__poly__degree": [1, 2, 3]
            
        }
    }
}

In [ ]:
for model, config in models.items():
    
    logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor ),
    ("model", config["model"])
    ])

In [ ]:
logistic_search = GridSearchCV(
                                logistic_pipeline, 
                                config["params"], 
                                n_jobs=-1,
                                cv=5, 
                                scoring="recall"
)

logistic_search.fit(X_train, y_train)

In [ ]:
best_logistic_params = logistic_search.best_params_
print(best_logistic_params)

best_logistic_model = logistic_search.best_estimator_
print(best_logistic_model)

EVALUATION

In [ ]:
y_pred = best_logistic_model.predict(X_test)

In [ ]:
## some evaluation metrics require probabilities

probabilities = best_logistic_model.predict_proba(X_test)[:,1]

In [ ]:

def evaluate(y_test, y_pred, probabilities):
  results = {
      "accuracy": accuracy_score(y_test, y_pred),
      "precision": precision_score(y_test, y_pred),
      "recall": recall_score(y_test, y_pred),
      "f1": f1_score(y_test, y_pred),
      "roc_auc": roc_auc_score(y_test, probabilities),
      "confusion_matrix": confusion_matrix (y_test, y_pred),
      "classification_report": classification_report(y_test, y_pred)
  }

  return results

In [ ]:
evaluate(y_test, y_pred, probabilities)

Confusion matrix

- True positives: as observed previosly the model has a few failures and it has captured most of them
- False positives: a few or no false machine_failures where predicted

- True negatives: all or most machine_failures were captured
- False negatives: a few non-machine_failures where predicted to be machine_failures which is better than getting false negatives

In [ ]:
ConfusionMatrixDisplay.from_estimator(
    best_logistic_model, 
    X_test, 
    y_test)
plt.title("Confusion Matrix")
plt.show()

ROC Curve

In [ ]:
RocCurveDisplay.from_estimator(
    best_logistic_model,
    X_test,
    y_test
)
plt.title("ROC Curve")
plt.show()

Precision-Recall Curve

In [ ]:
PrecisionRecallDisplay.from_estimator(
    best_logistic_model,
    X_test,
    y_test
    )
plt.title("Precision-Recall Curve")
plt.show()